In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

In [33]:
class model:
    def __init__(self,layers):
        self.layers = layers
        self.layer_length = len(self.layers)

        self.weights = []
        self.bias = []
        self._cache = []

        for i in range(self.layer_length - 1):
            w = np.random.randn(layers[i+1],layers[i]) * 1 / np.sqrt(layers[i])
            b = np.zeros((layers[i+1],1))

            self.weights.append(w)
            self.bias.append(b)

    def Relu(self,z):
        return np.maximum(0,z)
    
    def Relu_deriv(self,z):
        return (z>0).astype(float)
    
    def sigmoid(self,z):
        return 1/(1 + np.exp(-z))
    
    def softmax(self, z):
        exp_z = np.exp(z - np.max(z))
        
        return exp_z / np.sum(exp_z)
    
    @staticmethod
    def one_hot_encode(Y, num_classes=10):
        one_hot_Y = np.zeros((num_classes, Y.size))
        one_hot_Y[Y.astype(int), np.arange(Y.size)] = 1

        return one_hot_Y.astype(float)
    
    def forward(self,x):
        a_prev = x
        self.cache = []

        for i in range(len(self.weights)):
            z = self.weights[i] @ a_prev + self.bias[i]
            
            if(i == len(self.weights) - 1):
                a_l = self.softmax(z)
            else:
                a_l = self.Relu(z)
                

            self._cache.append((z, a_prev))
            a_prev = a_l
    
        return a_prev

    def backprop(self,y,y_hat):
        M = y_hat.shape[1]
        L = len(self.weights)
        grads = {}

        dZ = y_hat - y

        for i in reversed(range(L)):
            if( i == L-1):
                z_curr,a_prev = self._cache[i-1]
            else:
                z_curr,a_prev = self._cache[i]

            w_curr = self.weights[i]

            grads['dW' + str(i)] = (1 / M) * dZ @ a_prev.T
            grads['db' + str(i)] = (1 / M) * np.sum(dZ, axis=1, keepdims=True)

            if i != 0:
                dA = w_curr.T @ dZ
                dZ = dA * self.Relu_deriv(z_curr)

        return grads   

    def update_params(self,grad,learning_rate):
        for i in range(len(self.weights)):
            self.weights[i] -= learning_rate * grad['dW' + str(i)] 
            self.bias[i] -= learning_rate * grad['db' + str(i)]



In [34]:
def train(model,x_train,y_train_oh,learning_rate,epoch):
    M = len(x_train[0])

    for epochs in range(epoch):
        y_hat = model.forward(x_train)

        loss = - (1 / M) * np.sum(y_train_oh * np.log(y_hat))

        grads = model.backprop(y_train_oh,y_hat)   

        model.update_params(grads,learning_rate)

        if (epochs + 1) % 10 == 0:
            # Calculate accuracy to see performance
            predictions = np.argmax(y_hat, axis=0) # Get the index of the highest probability
            actual_labels = np.argmax(y_train_oh.T, axis=1) # Get the original class index
            accuracy = np.mean(predictions == actual_labels) * 100
            
            print(f"Epoch {epochs + 1}/{epochs} | Loss: {loss:.4f} | Accuracy: {accuracy:.2f}%")

In [35]:
(X_train, Y_train), (X_test, Y_test) = tf.keras.datasets.mnist.load_data()

# Flatten the 28x28 images into a 784-dimensional vector and transpose
X_train_flat = X_train.reshape(X_train.shape[0], -1).T
X_test_flat = X_test.reshape(X_test.shape[0], -1).T

# Normalize the pixel values (original range 0-255) to 0-1
X_train_norm = X_train_flat / 255.0
X_test_norm = X_test_flat / 255.0

Y_train_OH = model.one_hot_encode(Y_train)
Y_test_OH = model.one_hot_encode(Y_test)

In [36]:
my_model = model([784,10,10,10])
Learningrate = 0.1
epochs = 500

In [37]:
train(my_model, X_train_norm, Y_train_OH, Learningrate, epochs)

Epoch 10/9 | Loss: 13.7968 | Accuracy: 9.74%
Epoch 20/19 | Loss: 17.0063 | Accuracy: 9.74%
Epoch 30/29 | Loss: 26.2180 | Accuracy: 9.74%
Epoch 40/39 | Loss: 42.5077 | Accuracy: 9.74%
Epoch 50/49 | Loss: 66.6636 | Accuracy: 9.74%
Epoch 60/59 | Loss: 100.9901 | Accuracy: 9.74%
Epoch 70/69 | Loss: 149.0208 | Accuracy: 9.74%
Epoch 80/79 | Loss: 215.0906 | Accuracy: 9.74%
Epoch 90/89 | Loss: 306.7298 | Accuracy: 9.74%
Epoch 100/99 | Loss: 431.0059 | Accuracy: 9.74%
Epoch 110/109 | Loss: 597.9368 | Accuracy: 9.74%


C:\Users\Rayyan\AppData\Local\Temp\ipykernel_9872\1692978226.py:7: RuntimeWarning: divide by zero encountered in log
  loss = - (1 / M) * np.sum(y_train_oh * np.log(y_hat))
C:\Users\Rayyan\AppData\Local\Temp\ipykernel_9872\1692978226.py:7: RuntimeWarning: invalid value encountered in multiply
  loss = - (1 / M) * np.sum(y_train_oh * np.log(y_hat))


Epoch 120/119 | Loss: nan | Accuracy: 9.24%
Epoch 130/129 | Loss: nan | Accuracy: 7.33%
Epoch 140/139 | Loss: nan | Accuracy: 9.08%


KeyboardInterrupt: 